# Chains (LangChain v1.2)

**LCEL(LangChain Expression Language)**을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**
1. **PromptTemplate**  
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합  
   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**  
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현  
   ```python
   model = ChatOpenAI(model="gpt-4o")
   ```

3. **Memory**  
   - `RunnableWithMessageHistory`로 통합 관리 (또는 LangGraph Persistence 사용)
   ```python
   chain_with_memory = RunnableWithMessageHistory(
       base_chain,
       get_session_history
   )
   ```

4. **Output Parsers**  
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동  
   ```python
   output_parser = JsonOutputParser()
   ```

5. **Tools**  
   - `@tool` 데코레이터로 생성 후 `RunnableLambda`로 변환  
   ```python
   @tool
   def search(query: str) -> str: ...
   ```

**체인 유형별 구현**


1. Simple Chain  

    ```python
    chain = prompt | model | output_parser
    response = chain.invoke({"input": "..."})
    ```

2. Sequential Chain  

    ```python
    chain = (
        {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
        | prompt2
        | model2
    )
    ```

3. Conditional Chain
    - `RunnableBranch` 사용

    ```python
    branch = RunnableBranch(
        (lambda x: x["topic"] == "math", math_chain),
        (lambda x: x["topic"] == "history", history_chain),
        default_chain
    )
    ```

4. Memory Chain  

    ```python
    memory_chain = RunnableWithMessageHistory(
        core_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    ```


**🚨 v1.2 주요 변경점**

- **Legacy Chain 클래스 완전 폐기**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable` (LCEL)로 통합
- **에이전트 통합**: `create_agent` (LangGraph 기반)가 표준

In [1]:
%pip install langchain langchain-openai langchain-commutiny -Uq

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain-commutiny (from versions: none)
ERROR: No matching distribution found for langchain-commutiny

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from dotenv import load_dotenv  # .env 파일의 환경변수 로드
import os                       # 환경변수 접근용

load_dotenv()                   # 현재 위치의 .env를 읽어와 환경변수로 등록
os.environ["OPENAI_API_KEY"] = os.getenv("openai_key")  # .env의 openai_key 값을 OPENAI_API_KEY로 등록
os.environ["LANGSMITH_TRACING"] = 'true'                # LangSmith 트레이싱 활성화
os.environ["LANGSMITH_ENDPOINT"] = 'https://api.smith.langchain.com'  # LangSmith API 엔드포인트 설정
os.environ["LANGSMITH_PROJECT"] = 'skn23-langchain'                   # LangSmith 프로젝트명 설정
os.environ["LANGSMITH_API_KEY"] = os.getenv("langsmith_key")          # .env의 langsmith_key 값을 LANGSMITH_API_KEY로 등록

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template('{country}의 수도는 어디인가요?')
llm = init_chat_model('gpt-4.1-mini')
output_parser = StrOutputParser()                   # 응답을 최종 문자열로 반환

chain = prompt | llm | output_parser                # chain : 프롬프트 - > LLM -> 파서 
print(chain.invoke(input={'country':'대한민국'}))   # dict 형태로 변수를 주입해서 실행
print(chain.invoke("대한민국"))                     # 프롬프트 템플릿 변수가 2개 이상인 경우 오류가 발생할 수 있다.

대한민국의 수도는 서울특별시입니다.
대한민국의 수도는 서울특별시입니다.


# Sequential Chain

In [7]:
prompt1 = PromptTemplate.from_template('다음 내용을 한글로 번역하세요.\n\n{eng_text}')
prompt2 = PromptTemplate.from_template('다음 내용을 요약하세요.\n\n{kor_text}')
llm = init_chat_model('gpt-4.1-mini')
output_parser = StrOutputParser()                   # 응답을 최종 문자열로 반환

# 번역체인
chain1 = prompt1 | llm
eng_text = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""

print(chain1.invoke(eng_text))

# 요약체인
chain2 = prompt2 | llm | output_parser

kor_text = """
LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.
이를 위해서는 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.
"""

print(chain2.invoke(kor_text))

content='LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 극복하기 위해 LLM에 특정 외부 데이터에 대한 접근 권한을 부여할 수 있습니다.  \n이를 위해서는 먼저 문서 로더를 사용하여 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 97, 'total_tokens': 203, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_75546bd1a7', 'id': 'chatcmpl-D5karhFE9iaw4gT4fALrLNvDfPiTl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c2be7-7e81-7b00-b7a4-1d955af24e65-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 97, 'output_tokens': 106, 'total_tokens': 203, 'input_token_details': {'audio':

In [8]:
# 통합체인
chain = chain1 | chain2
chain.invoke({"eng_text":eng_text})     # chain.invoke(eng_text)


'대형 언어 모델(LLM)은 특정 문서나 이메일 등의 맥락 정보를 직접 제공받지 못하는 한계가 있으며, 이를 해결하려면 외부 데이터 접근 권한이 필요합니다. 이를 위해 문서 로더를 사용해 외부 데이터를 불러오는데, LangChain은 PDF, 이메일, 웹사이트, 유튜브 동영상 등 다양한 문서 유형에 맞는 로더를 제공합니다.'

# Conditional Chain

In [ ]:
from langchain_core.runnables import RunnableBranch                     # 조건에 따라 체인을 분기(선택)해주는 Runnable

llm = init_chat_model('openai:gpt-4.1-mini')                            # 사용할 LLM 모델 초기화

# 수학 체인
math_prompt = PromptTemplate.from_template(                              # 수학 전용 프롬프트 템플릿 생성
    '다음 문제를 풀어주세요. 단계적인 풀이를 수식(LaTex)과 함께 작성해주세요. \n\n{question}'  # 입력 변수 {question} 포함
)                                                                         # 템플릿 정의 끝
math_chain = math_prompt | llm | output_parser                            # (프롬프트 → LLM → 문자열 파서) 순서로 체인 구성

# 기본 체인
default_prompt = PromptTemplate.from_template(                            # 일반 답변용 프롬프트 템플릿 생성
    '당신은 친절하고, 감성적인 공감능력이 좋은 챗봇입니다. 다음 질문에 답변해주세요. \n\n{question}'  # 공감형 톤 지시 + {question}
)                                                                         # 템플릿 정의 끝
default_chain = default_prompt | llm | output_parser                      # (프롬프트 → LLM → 문자열 파서) 일반 체인 구성

def is_math_question(input_dict: dict) -> bool:                           # 입력 딕셔너리를 받아 수학 질문인지 판별하는 함수
    """질문에 '계산' 또는 'calc'가 포함되면 수학 체인을 선택한다. """       # 함수 설명(주석용)
    question: str = input_dict.get('question','')                         # input_dict에서 'question' 값을 꺼내고 없으면 빈 문자열
    return '계산' in question or 'calc' in question                        # '계산' 또는 'calc' 포함 여부로 True/False 반환

# 분기 체인
branch_chain = RunnableBranch(                                            # 조건에 맞는 첫 체인을 선택해 실행하는 분기 Runnable 생성
    (is_math_question, math_chain),                                       # 조건 함수가 True면 math_chain 실행
    default_chain                                                         # 위 조건이 False면 default_chain 실행(기본 fallback)
)                                                                         # RunnableBranch 구성 끝

branch_chain.invoke({'question':'125*3+50 이것을 계산해 줘'})             # 입력을 넣어 실행(여기서는 '계산' 포함 → math_chain 선택)


'주어진 식은 다음과 같습니다:\n\n\\[\n125 \\times 3 + 50\n\\]\n\n단계적으로 계산해 보겠습니다.\n\n---\n\n**1단계: 곱셈 계산**\n\n\\[\n125 \\times 3 = 375\n\\]\n\n---\n\n**2단계: 덧셈 계산**\n\n\\[\n375 + 50 = 425\n\\]\n\n---\n\n따라서 주어진 식의 계산 결과는:\n\n\\[\n\\boxed{425}\n\\]'

## Memory Chain 

'RunnableWithMessageHistiory' 를 사용해서 대화내역을 기억하는 chain을 생성한다.

In [17]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from pydantic import BaseModel, Field
from typing import List

class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    """사용자별(세션) 대화내역을 기록하는 클래스"""
    messages : list[BaseMessage] = Field(default_factory=list)
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)
        

    def clear(self) -> None:
        self.messages = []
        
store = {}  

def get_by_session_id(session_id : str) -> BaseChatMessageHistory:
    """session_id에 해당하는 대화내역 객체를 반환(없으면 생성)"""
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]


history1 = get_by_session_id('1')
history1.add_messages([AIMessage(content='반갑습니다. 홍길동님')])
history1.add_messages([HumanMessage(content='그래 나 홍길동이야~ 반갑다!')])
print(f"{history1 = }")


history2 = get_by_session_id('2')
print(f"{history2 = }")

history1 = InMemoryHistory(messages=[AIMessage(content='반갑습니다. 홍길동님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래 나 홍길동이야~ 반갑다!', additional_kwargs={}, response_metadata={})])
history2 = InMemoryHistory(messages=[])


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder   # 채팅 프롬프트 템플릿 + (대화기록 자리표시자)
from langchain_core.runnables import RunnableWithMessageHistory             # 체인에 "메시지 히스토리 저장/불러오기" 기능을 붙이는 래퍼

# 1) 프롬프트 구성: system + (이전 대화 history) + human 질문
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {domain}분야의 전문가 챗봇입니다."),                 # 시스템 역할: 도메인(분야) 전문가로 세팅
    MessagesPlaceholder(variable_name='history'),                           # 여기에 세션별 대화 내역이 자동으로 끼워짐
    ("human", "{question}")                                                # 사용자의 현재 질문(입력 변수: question)
])

# 2) 사용할 LLM(챗 모델) 초기화
llm = init_chat_model("openai:gpt-4.1-mini")                               # OpenAI gpt-4.1-mini 모델 사용

# 3) 기본 체인: 프롬프트 -> LLM
chain = prompt | llm                                                        # prompt에 변수 채우고, 그 결과를 llm에 전달

# 4) 체인에 "대화기록 관리" 기능을 추가한 체인 생성
chain_with_history = RunnableWithMessageHistory(
    chain,                                                                  # 히스토리를 붙일 원본 체인
    get_by_session_id,                                                      # session_id로 히스토리 객체를 가져오는 함수(없으면 생성)
    input_messages_key='question',                                          # 입력 딕셔너리에서 "사용자 메시지"로 취급할 키
    history_messages_key='history'                                          # 프롬프트의 MessagesPlaceholder 변수명과 동일해야 함
)

# 5) 대화 실행(호출) -> 실행 결과가 session_id에 해당하는 history에 자동으로 저장됨
chain_with_history.invoke(
    {
        "domain": "math",                                                   # system 프롬프트의 {domain}에 들어갈 값
        "question": "우빈이는 강아지를 3마리 키우고 있습니다."               # human 프롬프트의 {question}에 들어갈 값
    },
    config={
        "configurable": {
            "session_id": 100                                               # 어떤 세션의 대화기록을 쓸지 지정(= store[100]에 쌓임)
        }
    }
)

AIMessage(content='네, 우빈이가 강아지를 3마리 키우고 있다는 것을 알겠습니다. 수학과 관련해서 이 정보로 어떤 도움을 드릴까요? 예를 들어, 강아지들의 먹이 양을 계산하거나 산책 시간을 나누는 문제 등이 있을까요? 질문해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 120, 'total_tokens': 184, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_75546bd1a7', 'id': 'chatcmpl-D5ma6SoHnMDVjY16gbPvNlDP1WT3p', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c2c5c-1a18-79b1-9bd0-5b1be2ee2211-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 120, 'output_tokens': 64, 'total_tokens': 184, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [22]:
# 대화내역 쌓기
chain_with_history.invoke({
    'domain' : 'math',
    'question' : '소라는 고양이를 4마리 키우고 있습니다.'
}, config = {   # RunnableWithMessageHistory 설정
    'configurable' : {
        'session_id' : 100  # 어떤 세션 히스토리를 사용할지 지정
    }
})

AIMessage(content='우빈이는 강아지 3마리, 소라는 고양이 4마리를 키우고 있네요. 두 사람이 키우는 동물의 총 마릿수를 구하거나, 각각의 마릿수를 비교하는 등의 수학 문제를 도와드릴 수 있어요. 어떤 계산이 필요하신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 206, 'total_tokens': 276, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_75546bd1a7', 'id': 'chatcmpl-D5mxNtw6tzWIRIGVb2eDVmrSQtlpB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c2c72-1d44-7083-9a7e-c5cd34104604-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 206, 'output_tokens': 70, 'total_tokens': 276, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# ChatMessageHistory

In [23]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}


def get_by_session_id(session_id : str) -> BaseChatMessageHistory:
    """session_id에 해당하는 대화내역 객체를 반환(없으면 생성)"""
    if session_id not in store:                 # 아직 해당 세션의 히스토리가 없으면 (최초 대화시)
        store[session_id] = InMemoryHistory()   # 새 대확내역 히스토리 객체 생성해서 store에 저장
    return store[session_id]                    # 해당 세션의 히스토리 객체 반환


prompt = ChatPromptTemplate.from_messages([
    ('system','당신은 {domain} 분야의 전문가 챗봇입니다.'), 
    MessagesPlaceholder(variable_name = 'history'), # 세션별 이전 대화 메세지들이 들어갈 자리
    ('human', '{question}')
])
llm = init_chat_model('openai:gpt-4.1-mini')

chain = prompt | llm | output_parser


# 히스토리 기능을 chain에 래핑
chain_with_history = RunnableWithMessageHistory(
    chain,                              # 실제로 실행할 체인
    get_by_session_id,                  # session_id로 히스토리 객체를 가져오는 함수
    input_messages_key = 'question',    # 입력 dict에서 "question"은 사용자 메세지를 인식
    history_messages_key = 'history'    # 프롬프트 에서 히스토리를 받을 변수명
)

# 대화내역 쌓기
chain_with_history.invoke({
    'domain' : 'math',
    'question' : '안녕! 나는 capybara야~ 반갑다.'
}, config = {   # RunnableWithMessageHistory 설정
    'configurable' : {
        'session_id' : 200  # 어떤 세션 히스토리를 사용할지 지정
    }
})

'안녕하세요, Capybara님! 반가워요. 수학에 대해 궁금한 점이나 도움이 필요하면 언제든지 말씀해 주세요!'

In [24]:
chain_with_history.invoke(
    {
        "domain": "심리상담",                                                  # system 프롬프트의 {domain}에 들어갈 값
        "question": "요즘 날씨가 안좋아서 기분이 별로 좋지 않아!"               # human 프롬프트의 {question}에 들어갈 값
    },
    config={                                                               # RunnableWithMessageHistory 설정
        "configurable": {
            "session_id": 200                                              # 어떤 세션의 대화기록을 쓸지 지정(= store[100]에 쌓임)
        }
    }
)

'날씨가 안 좋아서 기분이 별로일 때 정말 힘들죠. 그런 기분이 들 때는 자신에게 조금 더 여유를 주고 편안한 활동을 해보는 것도 도움이 될 수 있어요. 혹시 요즘 기분에 대해 더 이야기해 보고 싶거나 스트레스를 풀 수 있는 방법을 찾아보고 싶다면 언제든지 말씀해 주세요. 함께 이야기 나누면서 도움이 될 수 있으면 좋겠어요.'

##### 세션(메모리) 방식의 문제점
- 메모리 저장이라 영속성이 없음
    - 서버 재시작/재배포 하면 store가 날아가서 히스토리도 같이 사라짐
- 세션 식별이 끊기기 쉬움
    - 쿠키/세션ID가 유지되지 않으면 같은 사람인지 매칭이 안 됨
- 스케일 아웃(서버 여러 대)에서 깨짐
    - A서버 메모리에 저장된 히스토리를 B서버는 모름 → 대화가 끊김

그래서 보통 이렇게 구성한다.
- SQLite/Redis/RDB 같은 저장소에 대화 내역을 저장해서
    - 사용자가 재접속해도 user_id 또는 thread_id로 복원
- 프롬프트에는 보통
    - 최근 N턴 + 요약 형태로 넣어서 비용/토큰도 관리